# OncoVision: extra lung models (2D nodule finder + benign vs malignant)

For when the 3D lung model was already trained by an earlier run. This prepares the same 641 patients with the same patient split (so the test patients are the same ones), then trains:
- a **2D nodule finder** for single CT images,
- an experimental **benign vs malignant nodule classifier**.

**How to run (important, so nothing gets lost):**
1. Right panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
2. Top right: **Save Version → Save & Run All (Commit) → Save**. Do *not* use the normal Run All.
3. Close the tab if you like. It runs in the background for about 1.5 hours.
4. Later open the notebook page → **Output** tab → download `oncovision_lung_extra.zip` and send it to Claude together with the printed `SCORES` line.

Data: LIDC-IDRI (The Cancer Imaging Archive, CC BY 3.0), via the public Hugging Face NIfTI conversion totalol/lungct. No login needed.

In [ ]:
!git clone -q -b vaishnavi/optimistic-noether-034odi https://github.com/Vaishnavi220506/Medical.git /kaggle/working/oncovision
%cd /kaggle/working/oncovision
!pip install -q -r requirements.txt huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings!")


## 1. Download and prepare the CTs (about 45 minutes)

In [ ]:
!python -m oncovision.training.train_lung3d --hf --prepared-dir /tmp/lung3d --val-cases 40 --test-cases 60 --workers 4 --prepare-only


## 2. 2D model for single CT images (about 30 minutes)
Slices of the same training patients; scored on slices of the held-out test patients.

In [ ]:
!python -m oncovision.training.export_slices --prepared-dir /tmp/lung3d --out /tmp/lung2d --split train --lesion-slices 6 --empty-slices 3
!python -m oncovision.training.export_slices --prepared-dir /tmp/lung3d --out /tmp/lung2d_test --split test --lesion-slices 6 --empty-slices 3
!python -m oncovision.training.train_segmenter --profile lung_ct --format maskfolder --data /tmp/lung2d/images --masks /tmp/lung2d/masks \
    --crop lungs --epochs 30 --batch-size 16 --workers 4 --out weights/lung_ct/segmenter.pt
!python -m oncovision.training.evaluate segmenter --profile lung_ct --format maskfolder --data /tmp/lung2d_test/images \
    --masks /tmp/lung2d_test/masks --weights weights/lung_ct/segmenter.pt --out /kaggle/working/lung2d_test


## 3. Benign vs malignant nodule classifier (about 15 minutes, experimental)

In [ ]:
!python -m oncovision.training.train_nodule_cls --prepared-dir /tmp/lung3d --epochs 40 --out weights/lung_ct/nodule_classifier.pt


## 4. Scores and package

In [ ]:
import torch, json, glob
scores = {}
for path in sorted(glob.glob("weights/lung_ct/*.pt")):
    meta = torch.load(path, map_location="cpu", weights_only=True)["meta"]
    scores[path.split("/")[-1]] = {k: {kk: vv for kk, vv in v.items() if kk != "by_threshold"} for k, v in meta.get("metrics", {}).items()}
for path in glob.glob("/kaggle/working/*2d_test/metrics.json"):
    scores[path.split("/")[-2]] = {k: v for k, v in json.load(open(path)).items() if not isinstance(v, (list, dict)) or k == "dice"}
print("SCORES (send these to Claude):", json.dumps(scores))
!python scripts/export_weights.py --src weights/lung_ct --dst export_weights/lung_ct
!cd /kaggle/working/oncovision/export_weights && rm -f /kaggle/working/oncovision_lung_extra.zip && zip -qr /kaggle/working/oncovision_lung_extra.zip lung_ct  && ls -la /kaggle/working/oncovision_lung_extra.zip
